## 地震属性与子波 gain 关系比较

本 notebook 读取 `wavelet_batch_synthetic_depth@20260427.ipynb` 的输出，按井内有效样点自适应切段，比较三个地震振幅属性与 gain 的关系。

当前比较的属性：

- `seismic_rms`
- `seismic_abs_mean`
- `seismic_abs_p90`

每个 segment 的 gain 在同一段内用最小二乘估计；属性和 gain 使用同一组 segment 窗口。


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 120)


## 1) 配置

In [ ]:
data_root = repo_root / "data"

wavelet_batch_output_dir = data_root / "output_wavelet_batch_synthetic_depth_20260428"
batch_metrics_file = wavelet_batch_output_dir / "wavelet_batch_metrics.csv"

output_dir = data_root / "output_wavelet_amplitude_gain_attribute_compare_20260429"
figure_dir = output_dir / "figures"
segment_samples_file = output_dir / "wavelet_amplitude_gain_attribute_segment_samples.csv"
relationship_metrics_file = output_dir / "wavelet_amplitude_gain_attribute_relationship_metrics.csv"
best_relationship_file = output_dir / "wavelet_amplitude_gain_attribute_best_relationship.csv"

attribute_columns = ["seismic_rms", "seismic_abs_mean", "seismic_abs_p90"]

# Adaptive segmentation parameters.
min_segment_valid_samples = 8
max_segment_count = 20
min_segments_per_well = 1
gain_eps = 1e-12

# Optional quality gate. Leave as None to include all successful batch synthetic wells.
min_batch_corr = None

required_paths = [batch_metrics_file]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

batch_metrics_df = pd.read_csv(batch_metrics_file)
required_metric_columns = {"well_name", "status", "corr", "scale", "synthetic_qc_path", "depth_shift_curve_path"}
missing = required_metric_columns - set(batch_metrics_df.columns)
if missing:
    raise ValueError(f"Batch metrics missing columns: {sorted(missing)}")

ok_batch_df = batch_metrics_df.loc[batch_metrics_df["status"] == "ok"].copy()
if min_batch_corr is not None:
    ok_batch_df = ok_batch_df.loc[ok_batch_df["corr"] >= float(min_batch_corr)].copy()
if ok_batch_df.empty:
    raise ValueError("No successful batch synthetic wells available for attribute comparison.")

print("Batch metrics:", batch_metrics_file)
print("Batch wells:", len(ok_batch_df))
print("Output dir:", output_dir)


## 2) 工具函数

In [ ]:
def resolve_artifact_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    return path if path.anchor else repo_root / path


def finite_pearson(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    if np.nanstd(x[mask]) <= 0.0 or np.nanstd(y[mask]) <= 0.0:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])


def finite_spearman(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    rank_x = pd.Series(x[mask]).rank(method="average").to_numpy(dtype=float)
    rank_y = pd.Series(y[mask]).rank(method="average").to_numpy(dtype=float)
    return finite_pearson(rank_x, rank_y)


def set_log_column(df: pd.DataFrame, source_column: str, log_column: str) -> None:
    values = df[source_column].to_numpy(dtype=float)
    df[log_column] = np.nan
    positive = values > 0.0
    df.loc[positive, log_column] = np.log(values[positive])


def split_valid_indices(
    valid_indices: np.ndarray,
    *,
    min_valid_samples: int,
    max_segments: int,
    min_segments: int,
) -> list[np.ndarray]:
    valid_indices = np.asarray(valid_indices, dtype=np.int64)
    if valid_indices.size < int(min_valid_samples):
        return []
    segment_count = int(min(max_segments, valid_indices.size // int(min_valid_samples)))
    segment_count = max(int(min_segments), segment_count)
    return [segment for segment in np.array_split(valid_indices, segment_count) if segment.size >= min_valid_samples]


def positive_ls_gain(
    seismic_values: np.ndarray,
    synthetic_raw_values: np.ndarray,
    *,
    eps: float,
    min_valid_samples: int,
) -> float:
    seismic_values = np.asarray(seismic_values, dtype=float)
    synthetic_raw_values = np.asarray(synthetic_raw_values, dtype=float)
    valid = np.isfinite(seismic_values) & np.isfinite(synthetic_raw_values)
    if int(valid.sum()) < int(min_valid_samples):
        return np.nan
    numerator = float(np.sum(seismic_values[valid] * synthetic_raw_values[valid]))
    denominator = float(np.sum(synthetic_raw_values[valid] ** 2))
    gain = numerator / (denominator + float(eps) * max(int(valid.sum()), 1))
    return float(gain) if np.isfinite(gain) and gain > 0.0 else np.nan


def segment_attribute_values(seismic_values: np.ndarray) -> dict:
    seismic_values = np.asarray(seismic_values, dtype=float)
    finite = np.isfinite(seismic_values)
    if not np.any(finite):
        return {"seismic_rms": np.nan, "seismic_abs_mean": np.nan, "seismic_abs_p90": np.nan}
    values = seismic_values[finite]
    abs_values = np.abs(values)
    return {
        "seismic_rms": float(np.sqrt(np.nanmean(values**2))),
        "seismic_abs_mean": float(np.nanmean(abs_values)),
        "seismic_abs_p90": float(np.nanpercentile(abs_values, 90.0)),
    }


## 3) 构建 segment 样本

In [ ]:
segment_rows = []

for _, row in ok_batch_df.iterrows():
    well_name = str(row["well_name"])
    scale = float(row["scale"])
    if not np.isfinite(scale) or scale <= 0.0:
        continue

    synthetic_qc_path = resolve_artifact_path(row["synthetic_qc_path"])
    depth_shift_curve_path = resolve_artifact_path(row["depth_shift_curve_path"])
    if not synthetic_qc_path.exists():
        raise FileNotFoundError(synthetic_qc_path)
    if not depth_shift_curve_path.exists():
        raise FileNotFoundError(depth_shift_curve_path)

    qc_df = pd.read_csv(synthetic_qc_path)
    required_qc_columns = {
        "twt_s",
        "seismic_norm",
        "synthetic_scaled",
        "eval_mask",
    }
    missing = required_qc_columns - set(qc_df.columns)
    if missing:
        raise ValueError(f"Synthetic QC {synthetic_qc_path} missing columns: {sorted(missing)}")

    depth_shift_df = pd.read_csv(depth_shift_curve_path)
    required_depth_columns = {"twt_s", "tvdss_m"}
    missing = required_depth_columns - set(depth_shift_df.columns)
    if missing:
        raise ValueError(f"Depth shift curve {depth_shift_curve_path} missing columns: {sorted(missing)}")

    twt_s = qc_df["twt_s"].to_numpy(dtype=float)
    seismic_norm = qc_df["seismic_norm"].to_numpy(dtype=float)
    synthetic_scaled = qc_df["synthetic_scaled"].to_numpy(dtype=float)
    synthetic_raw = synthetic_scaled / scale
    eval_mask_series = qc_df["eval_mask"]
    if eval_mask_series.dtype == bool:
        eval_mask = eval_mask_series.to_numpy()
    else:
        eval_mask = eval_mask_series.astype(str).str.lower().isin(["true", "1", "yes"]).to_numpy()

    depth_twt = depth_shift_df["twt_s"].to_numpy(dtype=float)
    depth_z = depth_shift_df["tvdss_m"].to_numpy(dtype=float)
    finite_depth = np.isfinite(depth_twt) & np.isfinite(depth_z)
    depth_twt = depth_twt[finite_depth]
    depth_z = depth_z[finite_depth]
    if depth_twt.size < 2:
        raise ValueError(f"Depth shift curve {depth_shift_curve_path} has too few finite samples.")
    order = np.argsort(depth_twt)
    depth_twt = depth_twt[order]
    depth_z = depth_z[order]

    finite = (
        eval_mask
        & np.isfinite(twt_s)
        & np.isfinite(seismic_norm)
        & np.isfinite(synthetic_raw)
    )
    valid_indices = np.flatnonzero(finite)
    segments = split_valid_indices(
        valid_indices,
        min_valid_samples=min_segment_valid_samples,
        max_segments=max_segment_count,
        min_segments=min_segments_per_well,
    )

    for segment_id, segment_indices in enumerate(segments):
        seg_twt = twt_s[segment_indices]
        seg_seismic = seismic_norm[segment_indices]
        seg_synthetic_raw = synthetic_raw[segment_indices]
        gain = positive_ls_gain(
            seg_seismic,
            seg_synthetic_raw,
            eps=gain_eps,
            min_valid_samples=min_segment_valid_samples,
        )
        if not np.isfinite(gain):
            continue
        attrs = segment_attribute_values(seg_seismic)
        seg_tvdss = np.interp(seg_twt, depth_twt, depth_z)
        segment_rows.append(
            {
                "well_name": well_name,
                "segment_id": int(segment_id),
                "n_valid_samples": int(segment_indices.size),
                "twt_min_s": float(np.nanmin(seg_twt)),
                "twt_max_s": float(np.nanmax(seg_twt)),
                "tvdss_min_m": float(np.nanmin(seg_tvdss)),
                "tvdss_max_m": float(np.nanmax(seg_tvdss)),
                "gain": gain,
                **attrs,
                "batch_corr": float(row["corr"]),
                "batch_scale": scale,
                "synthetic_qc_path": str(synthetic_qc_path),
                "depth_shift_curve_path": str(depth_shift_curve_path),
            }
        )

segment_samples_df = pd.DataFrame(segment_rows)
if segment_samples_df.empty:
    raise ValueError("No finite segment samples were generated.")

for column in ["gain", *attribute_columns]:
    set_log_column(segment_samples_df, column, f"log_{column}")

segment_samples_df.to_csv(segment_samples_file, index=False)
print("Saved", segment_samples_file)
display(segment_samples_df.head())
print("n_segments:", len(segment_samples_df), "n_wells:", segment_samples_df["well_name"].nunique())


## 4) 属性相关性比较

In [ ]:
metric_rows = []
for attribute_column in attribute_columns:
    x_column = f"log_{attribute_column}"
    x = segment_samples_df[x_column].to_numpy(dtype=float)
    y = segment_samples_df["log_gain"].to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    metric_rows.append(
        {
            "scope": "all_wells",
            "well_name": "__all__",
            "attribute": attribute_column,
            "n_samples": int(mask.sum()),
            "pearson": finite_pearson(x, y),
            "spearman": finite_spearman(x, y),
            "abs_pearson": abs(finite_pearson(x, y)) if np.isfinite(finite_pearson(x, y)) else np.nan,
            "abs_spearman": abs(finite_spearman(x, y)) if np.isfinite(finite_spearman(x, y)) else np.nan,
        }
    )

    for well_name, well_df in segment_samples_df.groupby("well_name"):
        xw = well_df[x_column].to_numpy(dtype=float)
        yw = well_df["log_gain"].to_numpy(dtype=float)
        mask_w = np.isfinite(xw) & np.isfinite(yw)
        pearson = finite_pearson(xw, yw)
        spearman = finite_spearman(xw, yw)
        metric_rows.append(
            {
                "scope": "per_well",
                "well_name": well_name,
                "attribute": attribute_column,
                "n_samples": int(mask_w.sum()),
                "pearson": pearson,
                "spearman": spearman,
                "abs_pearson": abs(pearson) if np.isfinite(pearson) else np.nan,
                "abs_spearman": abs(spearman) if np.isfinite(spearman) else np.nan,
            }
        )

relationship_metrics_df = pd.DataFrame(metric_rows)
relationship_metrics_df.to_csv(relationship_metrics_file, index=False)

all_relationships_df = relationship_metrics_df.loc[relationship_metrics_df["scope"] == "all_wells"].copy()
all_relationships_df = all_relationships_df.sort_values(["abs_pearson", "abs_spearman"], ascending=False)
best_relationship_df = all_relationships_df.head(1).copy()
best_relationship_df.to_csv(best_relationship_file, index=False)

print("Saved", relationship_metrics_file)
print("Saved", best_relationship_file)
display(all_relationships_df)


## 5) 图形 QC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), constrained_layout=True)
plot_df = all_relationships_df.copy()
plot_df["label"] = plot_df["attribute"].str.replace("seismic_", "", regex=False)

axes[0].bar(np.arange(len(plot_df)), plot_df["pearson"], color="tab:blue", alpha=0.75)
axes[0].axhline(0.0, color="black", lw=0.8)
axes[0].set_xticks(np.arange(len(plot_df)))
axes[0].set_xticklabels(plot_df["label"], rotation=35, ha="right")
axes[0].set_ylabel("Pearson r")
axes[0].set_title("log(attribute) vs log(gain)")
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].bar(np.arange(len(plot_df)), plot_df["spearman"], color="tab:green", alpha=0.75)
axes[1].axhline(0.0, color="black", lw=0.8)
axes[1].set_xticks(np.arange(len(plot_df)))
axes[1].set_xticklabels(plot_df["label"], rotation=35, ha="right")
axes[1].set_ylabel("Spearman rho")
axes[1].set_title("Rank relationship")
axes[1].grid(True, axis="y", alpha=0.25)

summary_fig = figure_dir / "qc_01_gain_attribute_relationship_bars.png"
fig.savefig(summary_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", summary_fig)

best_row = best_relationship_df.iloc[0]
best_attribute = str(best_row["attribute"])
fig, ax = plt.subplots(figsize=(6.8, 5.2), constrained_layout=True)
well_names = segment_samples_df["well_name"].drop_duplicates().tolist()
color_map = {well_name: plt.cm.tab10(idx % 10) for idx, well_name in enumerate(well_names)}  # type: ignore
for well_name, well_df in segment_samples_df.groupby("well_name"):
    ax.scatter(
        well_df[f"log_{best_attribute}"],
        well_df["log_gain"],
        s=28,
        alpha=0.75,
        color=color_map[well_name],
        label=well_name,
    )
ax.set_xlabel(f"log({best_attribute})")
ax.set_ylabel("log(gain)")
ax.set_title(f"Best all-well relationship: {best_attribute}")
ax.legend(loc="best", fontsize=7)
ax.grid(True, alpha=0.25)
best_fig = figure_dir / "qc_02_best_gain_attribute_relationship_scatter.png"
fig.savefig(best_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", best_fig)


## 6) 输出清单

In [ ]:
print("Segment samples:")
print(segment_samples_file)
print()
print("Metrics:")
print(relationship_metrics_file)
print(best_relationship_file)
print()
print("Figures:")
for path in sorted(figure_dir.glob("*.png")):
    print(path)
